In [ ]:
import torch
import torch.nn as nn
import numpy as np
import scipy.io as sio
import time
import os
import argparse
import h5py
import torchvision.transforms as T
import torch.nn.functional as F
from nmi_loss import normalized_cross_correlation
import matplotlib.pyplot as plt

from PIL import Image
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

# GPU
GPU_NUM = 0 #GPU number
device = torch.device(f'cuda:{GPU_NUM}' if torch.cuda.is_available() else 'cpu')
torch.cuda.set_device(device) # change allocation of current GPU
print ('Current cuda device ', torch.cuda.current_device()) # check   

In [ ]:
## Dataset #########################################################################
ds_num = 63

X_mat = sio.loadmat("2024_048_mrf.mat")
train_X = X_mat['MRF']
train_X=np.transpose(train_X,(2,3,0,1))

X2_mat = sio.loadmat("mask_invivoMRF_048.mat")
train_X2 = X2_mat['mask']
train_X2=np.transpose(train_X2,(2,0,1))

X_train=torch.FloatTensor(train_X)
X2_train=torch.FloatTensor(train_X2)

mask = X2_train.unsqueeze(1)  # [11, 1, 256, 256]
X_train = X_train * mask   # broadcasting 

print(X_train.size())
print(X2_train.size())

#####################################################################################
trainset = TensorDataset(X_train,X2_train)

trainloader=DataLoader(trainset,batch_size=1,shuffle=True)
testloader=DataLoader(trainset,batch_size=1,shuffle=False)

#####################################################################################

In [ ]:
class RegNet(nn.Module):
    def __init__(self, num_ds):
        super().__init__()
        self.num_ds = num_ds

        # =============================STN=======================================
        # Spatial transformer localization-network
        self.localization = nn.Sequential(
            nn.Conv2d(2, 32, kernel_size=5, stride=1, padding=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=5, stride=1, padding=2),
            nn.MaxPool2d(2),  # H/2
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
            nn.MaxPool2d(2),  # H/4
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4))  # Output: [B, 32, 4, 4]
        )

        # Regressor for the 3 * 2 affine matrix
        self.fc_loc = nn.Sequential(
            nn.Linear(32 * 4 * 4, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 3)  # (angle, tx, ty)
        )

        # Initialize the weights/bias with identity transformation
        self.fc_loc[2].weight.data.zero_()
        self.fc_loc[2].bias.data.copy_(torch.tensor([0., 0., 0.], dtype=torch.float))

    
    def stn(self, x):
        h=x.size(2)
        w=x.size(3)
        for i in range(self.num_ds-1):
            xi = torch.cat((x[:,0,:,:].view(-1,1,h,w), x[:,i+1,:,:].view(-1,1,h,w)),1) # (B, C, H, W)
            xs = self.localization(xi)
            xs = xs.view(-1, 512) #
            params = self.fc_loc(xs)
            angle = params[:, 0]
            tx = params[:, 1]
            ty = params[:, 2]

            cos = torch.cos(angle)
            sin = torch.sin(angle)

            theta = torch.zeros(x.size(0), 2, 3, device=x.device)
            theta[:, 0, 0] = cos
            theta[:, 0, 1] = -sin
            theta[:, 1, 0] = sin
            theta[:, 1, 1] = cos
            theta[:, 0, 2] = tx
            theta[:, 1, 2] = ty
            
            if i==0:
                theta_a = theta.view(-1,1,2,3)
            else:
                theta_a = torch.cat((theta_a,theta.view(-1,1,2,3)),1)

        grid = F.affine_grid(theta_a.view(-1,2,3), x[:,1:,:,:].reshape(-1,1,h,w).size(),align_corners=True) # (N,C,H,W)
        motion_corrected_S_b = F.grid_sample(x[:,1:,:,:].reshape(-1,1,h,w), grid, mode='bilinear', padding_mode='reflection',align_corners=True) # (N,C,H,W)

        motion_corrected_S_b = motion_corrected_S_b.view(-1,self.num_ds-1,h,w)
        motion_corrected_S = torch.cat((x[:,0,:,:].view(-1,1,h,w), motion_corrected_S_b),1)

        theta_a = theta_a.view(-1, (self.num_ds-1), 2, 3)
        return motion_corrected_S, theta_a

    def forward(self, x, S0mask):

        #STN
        motion_corrected_S, theta = self.stn(x)
        
        motion_corrected_S_masked = motion_corrected_S * S0mask.unsqueeze(1)

        return motion_corrected_S_masked, theta

In [ ]:
epochs = 200

## Model training
model = RegNet(ds_num)
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)


In [ ]:
start_time_0epoch=time.time()
best_train_loss = 1000
for itr in range(epochs):
    batch_loss=0
    for i,data in enumerate(trainloader):
        [X_batch,mask_batch]=data
        X_batch = X_batch.to(device)
        mask_batch = mask_batch.to(device)

        output,theta = model(X_batch,mask_batch)
        
        loss = 0
        for bi in range(1, ds_num):
            loss += -torch.log((normalized_cross_correlation(X_batch[:,0,:,:], output[:,bi,:,:],False)+1)/2)

         
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        batch_loss += loss.item()
    
    if best_train_loss>batch_loss:
        best_train_loss=batch_loss
        best_model=model
        best_theta = theta

        
    if (itr+1)%10 == 0:
        print("iteration %d, loss = %.6f" % (itr+1, loss.item()))
        print('Time taken =', '{:.3f}'.format(time.time()-start_time_0epoch))  
        

In [ ]:
Registered_images = np.zeros(X_train.size())

for i,data in enumerate(testloader):
    [X_batch,mask_batch]=data
    X_batch = X_batch.to(device)
    mask_batch = mask_batch.to(device)

    idxs=list(range(i,(i+1)))

    output,theta = best_model(X_batch,mask_batch)
    Registered_images[idxs,:,:,:] += output.squeeze().detach().cpu().numpy()
    
sio.savemat("input_invivoMRF_loas_048_Registered.mat",{'input_invivo_test': Registered_images})

In [ ]:
X_train=torch.FloatTensor(Registered_images)

print(X_train.size())

#####################################################################################
trainset = TensorDataset(X_train)

trainloader=DataLoader(trainset,batch_size=1,shuffle=True)
testloader=DataLoader(trainset,batch_size=1,shuffle=False)
#####################################################################################

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_channel, mid_channel, out_channel, p = 0.3):
        super(ResBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channel, mid_channel, kernel_size = 3, padding = 1)
        self.conv2 = nn.Conv2d(mid_channel, out_channel, kernel_size = 3, padding = 1)
        # self.nonlinear1 = nn.LeakyReLU(0.1)
        self.nonlinear1 = nn.ReLU()
        self.Dropout = nn.Dropout(p)
        self.BN1 = nn.BatchNorm2d(num_features=mid_channel)
        self.BN2= nn.BatchNorm2d(num_features=out_channel)
        
    def forward(self, x):
        out1 = self.BN1(self.conv1(x))
        out2 = self.nonlinear1(out1)
        out3 = self.BN2(self.conv2(self.Dropout(out2)))
        out = x + out3
        
        return out

class self2self(nn.Module):
    def __init__(self, in_channel,out_channel, p):
        super(self2self, self).__init__()
        
        self.n_blks = 10
        self.channel_size = 64
        layers = [ResBlock(self.channel_size, self.channel_size, self.channel_size,p=p)]
        for i in range(self.n_blks-1):
            layers.append(ResBlock(self.channel_size, self.channel_size, self.channel_size,p=p))
        
        self.convs = nn.Sequential(*layers)
        self.conv_first = nn.Conv2d(in_channel, self.channel_size, kernel_size = 3, padding = 1)
        self.conv_last = nn.Conv2d(self.channel_size, out_channel, kernel_size = 3, padding = 1)
        self.conv_last2 = nn.Conv2d(out_channel, out_channel, kernel_size = 1)
        self.sig = nn.Sigmoid()
        self.relu = nn.ReLU()

    def forward(self, x):
        
        out1 = self.conv_first(x)
        out2 = self.convs(out1)
        out2 += out1
        out3 = self.conv_last(out2)
        out = self.conv_last2(out3)

        out = self.relu(out)
                
        return out

In [ ]:
def data_augmentation(img, flip_v, flip_h,device):
    axis = []
    if flip_v:
        axis.append(2)
    if flip_h:
        axis.append(3)
    if len(axis):
        img = torch.flip(img, axis)
        
    return img

In [ ]:
mask_p=0.3
dropout_p=0.3
learning_rate =1e-4
epochs = 10000

## Model training
model = self2self(ds_num,ds_num,dropout_p)
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
start_time_0epoch=time.time()
for itr in range(epochs):
    batch_loss=0
    for i,data in enumerate(trainloader):
        [X_batch]=data
        X_batch_norm = X_batch
        X_batch_norm = X_batch_norm.to(device)
        
        flip_v, flip_h = np.random.choice(2, size=2)
        Aug_input = data_augmentation(X_batch_norm, flip_v, flip_h,device)
            
        p_mtx = np.random.uniform(size=X_batch_norm.shape)
        mask = (p_mtx>mask_p).astype(np.double)
        mask = torch.tensor(mask).to(device, dtype=torch.float32)
        
        X_input = Aug_input
        y = Aug_input
        
        model.train()
        img_input_tensor = X_input*mask
        output = model(img_input_tensor)
         
        loss = torch.sum((output-y)*(output-y)*(1-mask))/torch.sum(1-mask)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        batch_loss += loss.item()
        
    if (itr+1)%1000 == 0:
        print("iteration %d, loss = %.10f" % (itr+1, loss.item()))
        print('Time taken =', '{:.3f}'.format(time.time()-start_time_0epoch))  
        torch.save(model.state_dict(),'checkpoints/Self2self_invivoMRF_loas_048_{}epochs.pth'.format(itr+1))
        

In [ ]:
NPred=10
sum_preds = np.zeros(X_train.size())

for i,data in enumerate(testloader):
    [X_batch]=data
    X_batch_norm = X_batch
    X_batch_norm = X_batch_norm.to(device)
    
    idxs=list(range(i,(i+1)))
    
    for j in range(NPred):
        p_mtx = np.random.uniform(size=X_batch_norm.shape)
        mask = (p_mtx>mask_p).astype(np.double)
        mask = torch.tensor(mask).to(device, dtype=torch.float32)
        
        img_input = X_batch_norm*mask
        
        output_test = model(img_input)
        sum_preds[idxs,:,:,:] += output_test.squeeze().detach().cpu().numpy()
    
Denoised_images=sum_preds/NPred

sio.savemat("input_invivoMRF_loas_048_Registered_Denoised.mat",{'input_invivo_test': Denoised_images})
